# 02 — Attach political party

Read the v1 parquet built by `src/preprocessing.py`, match `page_name` and `bylines` against the 2022 AEC candidate list, add a `political_party` column (`PartyAb` or null), and write a new v2 parquet partitioned by party.

## 1. Spark session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('FB_API_party_match') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

## 2. Paths

In [ ]:
IN_PATH        = '/user/s3348393/main/preprocessing/v1/parquet'
OUT_PATH       = '/user/s3348393/main/preprocessing/v2/parquet'
CANDIDATES_CSV = '../data/2022_election_candidates.csv'

## 3. Load v1 parquet

In [ ]:
df = spark.read.parquet(IN_PATH)
print('Rows:', df.count())
df.printSchema()

## 4. Load 2022 candidates

Columns: `DivisionNm, PartyAb, PartyNm, Surname, GivenNm`. Names are uppercase; surnames can be multi-word (`ABDUL RAZAK`), given-name field can be empty.

In [ ]:
candidates = spark.read.csv(CANDIDATES_CSV, header=True).toPandas()
print('Candidates:', len(candidates))
candidates.head()

## 5. Tokenise `page_name` and `bylines`

Use Spark MLlib `RegexTokenizer` (splits on `\W+`, lowercases) + `StopWordsRemover` with English defaults extended by political honorifics. Fields are tokenised independently — we won't concatenate them before matching, to avoid a given name in `page_name` crossing with a surname in `bylines` to spuriously match a candidate.

`coalesce(col, lit(''))` because `RegexTokenizer` errors on null inputs.

In [ ]:
from pyspark.sql.functions import coalesce, col, lit
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

df = df.withColumn('page_name_safe', coalesce(col('page_name'), lit(''))) \
       .withColumn('bylines_safe',   coalesce(col('bylines'),   lit('')))

honorifics = ['mp', 'hon', 'dr', 'mr', 'mrs', 'ms', 'sen', 'senator', 'rt', 'authorised', 'by', 'for']
stopwords = StopWordsRemover.loadDefaultStopWords('english') + honorifics

for src, tok_col, term_col in [
    ('page_name_safe', 'page_name_tokens', 'page_name_terms'),
    ('bylines_safe',   'bylines_tokens',   'bylines_terms'),
]:
    df = RegexTokenizer(inputCol=src, outputCol=tok_col, pattern=r'\W+', toLowercase=True).transform(df)
    df = StopWordsRemover(inputCol=tok_col, outputCol=term_col, stopWords=stopwords).transform(df)

df.select('page_name', 'page_name_terms', 'bylines', 'bylines_terms').show(5, truncate=80)

## 6. Build candidate match index

For each candidate, the *required tokens* are the first given-name token plus all surname tokens — e.g. Adam ABDUL RAZAK → `{adam, abdul, razak}`. Token-set subset matching is order-insensitive, so 'Adam Abdul Razak' and 'Abdul Razak, Adam' both match.

Index by the first surname token to keep matching fast: at lookup time we only check candidates whose key token appears in the ad's token list, rather than scanning all ~1,500 candidates per ad.

In [ ]:
from collections import defaultdict
import re

def split_tokens(s):
    if s is None:
        return []
    return [t for t in re.split(r'\W+', s.lower()) if t]

candidate_index = defaultdict(list)
skipped = 0
for _, row in candidates.iterrows():
    surname_toks = split_tokens(row['Surname'])
    given_toks   = split_tokens(row['GivenNm'])
    if not surname_toks:
        skipped += 1
        continue
    required = frozenset(surname_toks + ([given_toks[0]] if given_toks else []))
    candidate_index[surname_toks[0]].append((required, row['PartyAb']))

print('Index keys:', len(candidate_index))
print('Total candidate records:', sum(len(v) for v in candidate_index.values()))
print('Skipped (no surname):', skipped)

## 7. Assign `political_party`

For each ad: look up `page_name_terms` and `bylines_terms` independently. A candidate matches a field if its required-token set is a subset of that field's tokens. Collect the set of matched `PartyAb`s across both fields — if it resolves to exactly one party, that's the label; otherwise null (conservative: ambiguous matches stay unlabelled).

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

_INDEX = dict(candidate_index)  # serialise plain dict, not defaultdict

def _match_field(terms, index):
    if not terms:
        return set()
    term_set = set(terms)
    parties = set()
    for tok in term_set:
        for required, party in index.get(tok, ()):
            if required.issubset(term_set):
                parties.add(party)
    return parties

@udf(StringType())
def match_party(page_terms, bylines_terms):
    parties = _match_field(page_terms, _INDEX) | _match_field(bylines_terms, _INDEX)
    if len(parties) == 1:
        return next(iter(parties))
    return None

df = df.withColumn('political_party', match_party('page_name_terms', 'bylines_terms'))

## 8. Sanity checks

In [ ]:
df.groupBy('political_party').count().orderBy(col('count').desc()).show(50, truncate=False)

In [ ]:
# spot-check: a few matched rows per top party
from pyspark.sql.functions import desc

top_parties = [r['political_party'] for r in
               df.filter(col('political_party').isNotNull())
                 .groupBy('political_party').count()
                 .orderBy(desc('count')).limit(5).collect()]

for p in top_parties:
    print(f'\n=== {p} ===')
    df.filter(col('political_party') == p) \
      .select('page_name', 'bylines', 'political_party') \
      .show(10, truncate=60)

In [ ]:
# eyeball false positives: highest-spend matched rows
df.filter(col('political_party').isNotNull() & col('spend_mid').isNotNull() & (col('ad_seq_no') == 1)) \
  .orderBy(col('spend_mid').desc()) \
  .select('page_name', 'bylines', 'political_party', 'spend_mid') \
  .show(20, truncate=60)

In [ ]:
# eyeball false negatives: high-volume political-looking bylines that stayed null
from pyspark.sql.functions import lower as sql_lower

df.filter(col('political_party').isNull() & sql_lower(col('bylines')).contains('authorised by')) \
  .groupBy('bylines').count() \
  .orderBy(desc('count')) \
  .show(20, truncate=80)

## 9. Write v2 parquet partitioned by party

Drop the intermediate token/term columns before writing — they're large arrays and easy to rebuild. Rows with `political_party = null` land in `political_party=__HIVE_DEFAULT_PARTITION__/` (expected; that's the bulk of the corpus).

In [ ]:
intermediate = ['page_name_safe', 'bylines_safe',
                'page_name_tokens', 'bylines_tokens',
                'page_name_terms', 'bylines_terms']

out = df.drop(*intermediate)

out.write.partitionBy('political_party').parquet(OUT_PATH, mode='overwrite')
print('Wrote:', OUT_PATH)

In [ ]:
# round-trip check
rt = spark.read.parquet(OUT_PATH)
print('Roundtrip rows:', rt.count())
rt.printSchema()